<a href="https://colab.research.google.com/github/e23189uop/Statistical-Learning-e23189/blob/main/assignment%2307c_e23189/assignment07c_e23189.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here is a complete, rewritten version of the provided notebook's questions, mathematical derivations, and Python code blocks.

### I. Bayesian Estimation of a User Ability Parameter from Item Responses

**Problem Statement**
An educational platform administers a sequence of $n$ multiple-choice questions one at a time. The platform dynamically estimates the user's latent ability ($\Theta = \theta$) after every correct ($Y_i = 1$) or incorrect ($Y_i = 0$) response. The response probability is governed by a two-parameter logistic (2PL) model:


$$P(Y_i=1 \mid \Theta=\theta) = p_i(\theta) = \frac{1}{1+e^{-a_i(\theta-b_i)}}$$


Here, $a_i > 0$ represents the item's discrimination and $b_i$ denotes its difficulty.

Given a running history vector of responses $\mathbf{y}^{(k)} = (y_1, \dots, y_k)$, and a standard normal initial prior $\Theta \sim \mathscr{N}(0,1)$, evaluate the model through visualization, sequential likelihood formulations, mathematical updates, and numerical grid implementation.

**Sample Answers & Code**

**1. The 2PL Probability Model and Visualization**
Assuming conditional independence given $\Theta = \theta$, the general likelihood for a sequence of responses is:


$$P(Y=y \mid \Theta=\theta) = \prod_{i=1}^n [p_i(\theta)]^{y_i} [1-p_i(\theta)]^{1-y_i}$$



In [ ]:

import numpy as np
import plotly.graph_objects as go

# 2PL Item Response Function[cite: 1]
def calculate_p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta_range = np.linspace(-6, 6, 300)
curve_configs = [
    {"a": 0.5, "b": 0, "style": "dash"},
    {"a": 1.5, "b": -2, "style": "solid"},
    {"a": 1.5, "b": 0, "style": "solid"},
    {"a": 1.5, "b": 2, "style": "solid"},
] #[cite: 1]

fig = go.Figure()
for config in curve_configs:
    probs = calculate_p_i(theta_range, config["a"], config["b"])
    fig.add_trace(go.Scatter(
        x=theta_range, y=probs, mode='lines',
        name=f"a = {config['a']}, b = {config['b']}",
        line=dict(dash=config["style"], width=2.5)
    )) #[cite: 1]

fig.update_layout(
    title="2PL Item Response Probability Curves",
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y_i=1|θ)",
    template="plotly_white"
) #[cite: 1]
fig.show()





**2. Sequential Posterior Distribution**
For a single response $y_k$ at step $k$, the likelihood contribution is $L(y_k \mid \theta) = p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k}$. Using the previous step's posterior as the new prior, the recursive posterior density update is:


$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{ \left[ p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k} \right] f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) }{ \int_{\mathbb R} \left[ p_k(s)^{y_k} (1 - p_k(s))^{1 - y_k} \right] f_{\Theta \mid \mathbf{Y}^{(k-1)}}(s \mid \mathbf{y}^{(k-1)})\,ds }$$

**3. Point Estimations (Bayes and MAP)**

* **Running Posterior Mean (Bayes Estimate):** Minimizes squared-error loss and computes the expected value of the posterior: $\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \int_{\mathbb R} \theta \, f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})\,d\theta$.


* **MAP Estimate:** Finds the mode (peak) of the posterior distribution: $\widehat{\theta}_{\mathrm{MAP}}^{(k)} \in \arg\max_{\theta \in \mathbb R} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$.




In [ ]:

import scipy.stats as stats

# PART 1: Manual 4-Item Simulation Update
theta_grid = np.linspace(-5, 5, 500)
posterior = stats.norm.pdf(theta_grid, 0, 1) # Initial Prior

item_sequence = [
    {"a": 1.0, "b": -1.5, "y": 1},
    {"a": 1.5, "b": 0.5,  "y": 1},
    {"a": 1.2, "b": 1.5,  "y": 0},
    {"a": 2.0, "b": 0.2,  "y": 1}
]

# In practice, loop over item_sequence, multiply likelihood by posterior,
# and use np.trapezoid to normalize the density array




### II. Bayesian Tracking of Click-Through Rates (CTR)

**Problem Statement**
An e-commerce engine dynamically estimates advertisement CTRs one impression at a time. The true CTR $\Theta = \theta$ is modeled as a Bernoulli trial where a click is $Y_k = 1$ and a non-click is $Y_k = 0$. The platform utilizes a Beta distribution as the initial prior over $[0, 1]$. Plot the Beta distributions, derive the closed-form Beta-Binomial conjugate update, and simulate this mathematically.

**Sample Answers & Code**

**1. Sequential Joint Likelihood**
For an isolated response, the likelihood is $L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$. The joint likelihood for the history vector, assuming conditional independence, simplifies to:


$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{C_k} (1 - \theta)^{k - C_k}$$


where $C_k$ is the total observed clicks.

**2. Conjugate Updates and Posterior Mean**
Applying Bayes' Theorem recursively, we drop the normalizing denominator to get:


$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$


Combining exponents proves the Beta-Binomial conjugacy:


$$\alpha_k = \alpha_{k-1} + y_k \quad \text{and} \quad \beta_k = \beta_{k-1} + (1 - y_k)$$


The closed-form Posterior Mean at step $k$ evaluates to:


$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k} = \frac{\alpha_0 + C_k}{(\alpha_0 + \beta_0) + k}$$



In [ ]:
# Sequential Beta-Binomial Update Simulation
np.random.seed(42)
true_ctr = 0.35
alpha_current, beta_current = 1, 1 # Initial uniform prior

bayes_estimates = [alpha_current / (alpha_current + beta_current)] #

for k in range(1, 101):
    # Simulate stochatic interaction
    y_k = 1 if np.random.uniform(0, 1) < true_ctr else 0

    # Exact Conjugate Update
    alpha_current += y_k
    beta_current += (1 - y_k)

    bayes_estimates.append(alpha_current / (alpha_current + beta_current)) #


### III. Gaussian Mixture Clustering as Conditional Updating

**Problem Statement**
Consider observations $x_1, \dots, x_n$ to be clustered into $K$ groups using a latent random variable $C_i \in \{1, \dots, K\}$. Prove the formulation of the Gaussian mixture density, derive the posterior cluster probabilities (responsibilities), demonstrate the transition from soft to hard clustering via expected values, and outline the Expectation-Maximization (EM) log-likelihood updates.

**Sample Answers**

**1. Marginal Density & Responsibilities**
Using the Law of Total Probability, the marginal density is formed by mixing Gaussian components parameterized by mixture weights $\phi_k$:


$$p(x_i) = \sum_{k=1}^K \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$$


By Bayes' rule, the posterior probability (responsibility $\gamma_{ik}$) of cluster membership dynamically updates given a specific observation:


$$\gamma_{ik} = P(C_i=k \mid X_i=x_i) = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)}$$

**2. Soft Assignments via Conditional Expectation**
Defining a one-hot encoded vector $Z_i$ for cluster identities, the expected value of this vector acts precisely as the soft assignment vector (responsibilities):


$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(Z_{ik} = 1 \mid X_i = x_i) = P(C_i = k \mid X_i = x_i) = \gamma_{ik}$$


While soft clustering retains probabilistic ambiguity ($\gamma_{ik} \in [0, 1]$), hard clustering forces a deterministic assignment by taking $\widehat C_i = \operatorname{arg\,max}_{1 \le k \le K} \gamma_{ik}$.

**3. Complete-Data Likelihood & The Q-Function**
If latent labels $z_{ik}$ were observed constants, the complete-data log-likelihood $\ell_c$ simplifies into independent maximization problems per cluster:


$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$


Because $z_{ik}$ is unknown, the EM algorithm substitutes it with its conditional expectation ($\gamma_{ik}$) during the E-step, forming the $Q$-function:


$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$


During the M-step, the model updates structural parameters (like the means $\mu_k = \frac{1}{N_k} \sum \gamma_{ik} x_i$) weighted heavily by these posterior membership probabilities.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        self.scaler = StandardScaler()
        self.model = GaussianMixture(
            n_components=n_components,
            covariance_type="full",
            random_state=random_state,
        )

    def prepare_data(self, df, feature_cols, test_size=0.2):
        X = df[feature_cols].dropna().values
        X_scaled = self.scaler.fit_transform(X) # Ensure variance shapes properly
        return train_test_split(X_scaled, test_size=test_size, random_state=42)

    def fit(self, X_train):
        self.model.fit(X_train)
        print(f"Converged: {self.model.converged_}")
